In [2]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [3]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

In [4]:
MODEL_PATH = PROJECT_ROOT / "models" / "efficientnet_v2_finetuned.h5"
model = tf.keras.models.load_model(MODEL_PATH)

In [15]:
def crop_brain(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)

    contours, _ = cv2.findContours(
        thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        return image

    c = max(contours, key=cv2.contourArea)

    x, y, w, h = cv2.boundingRect(c)

    cropped = image[y:y+h, x:x+w]

    return cropped


# =========================
# 3. PREPROCESS FUNCTION
# =========================
def preprocess_mri(image_path):
    # Read image
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError("Image not found or path incorrect")

    # BGR → RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Crop brain
    img = crop_brain(img)

    # Resize (MUST match training)
    img = cv2.resize(img, (224, 224))

    # Convert to float32
    img = img.astype(np.float32)

    # EfficientNet preprocessing
    img = preprocess_input(img)

    # Add batch dimension
    img = np.expand_dims(img, axis=0)

    return img


# =========================
# 4. PREDICTION FUNCTION
# =========================
def predict_mri(image_path):
    img = preprocess_mri(image_path)

    pred = model.predict(img)

    print("Raw prediction:", pred)

    # Binary classification
    if pred[0][0] > 0.5:
        return "Healthy"
    else:
        return "Epilepsy detected"



In [21]:

image_path = r"D:\EpilepsyNexus\EpilepsyNexus\trails\h2.png"
result = predict_mri(image_path)

print("Final Result:", result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
Raw prediction: [[0.7617154]]
Final Result: Healthy
